# East West Segmentation Breach Risk
### Network Traffic Intrusion Analytics — World-Class Investigative AI Notebook

---

**Notebook file:** `network_traffic_intrusion_analytics/07_east_west_segmentation_breach_risk.ipynb`

**Why this matters:** Analysts use this workflow to make **east west segmentation breach risk** faster, more explainable, and easier to defend during investigative handoff. The notebook is deliberately structured to preserve analyst intuition, compare against a baseline, and expose where the model can fail.

## Investigative Context

| Dimension | Details |
| --- | --- |
| Topic focus | Traffic, flow, and protocol analytics for intrusion triage, lateral movement, and exfiltration hunting. |
| Investigative question | How can investigators use AI to make **east west segmentation breach risk** more consistent and auditable? |
| Operational decision | whether to isolate hosts, block destinations, or expand packet capture scope |
| Target label | `case_priority` |
| Learning goal | Learn how to go from evidence profiling to baseline comparison, explainability, and analyst handoff without losing forensic context. |

## Evidence Sources

| Source | Why It Matters | Caveat |
| --- | --- | --- |
| NetFlow / Zeek summaries | Volume shifts, beacon periodicity, and east-west movement | Sampling can undercount short sessions |
| DNS and proxy logs | Domain generation, tunneling, and egress path changes | Resolver caching changes frequency signatures |
| TLS / JA3 fingerprints | Client impersonation and unusual tooling families | Shared libraries can collide on fingerprints |
| Authentication and VPN logs | Impossible travel, remote access spikes, and account abuse | Burstiness around maintenance windows is normal |

## Standards and References

- MITRE ATT&CK
- Zeek protocol logging guidance
- NIST SP 800-61

## Step 0 - Imports and Case Framing

Set up the notebook and make the modeling contract explicit before touching evidence.

In [ ]:
# Uncomment in Colab if needed:
# !pip install numpy pandas scikit-learn matplotlib seaborn scipy

from __future__ import annotations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

RNG = np.random.default_rng(42)
N_SAMPLES = 1800
NOTEBOOK_TITLE = 'East West Segmentation Breach Risk'
TOPIC_NAME = 'Network Traffic Intrusion Analytics'
TASK_TYPE = 'binary'
TARGET = 'case_priority'
CLASS_LABELS = ['routine_review', 'priority_escalation']
FOCUS_LABEL = 'priority_escalation'
FOCUS_LABEL_INDEX = 1
NUMERIC_FEATURES = ['failed_auth_count', 'unique_destination_ips', 'rdp_session_minutes', 'packet_size_variance', 'port_sweep_ratio', 'lateral_hop_count']
CATEGORICAL_OPTIONS = {
    'protocol_family': ['dns', 'tls', 'rdp', 'http', 'smb'],
    'network_zone': ['user_vlan', 'server_vlan', 'dmz', 'ot_segment']
}
ENGINEERED_FEATURES = ['signal_fusion_index', 'context_gap_score', 'stability_ratio']
MODEL_FEATURES = ['failed_auth_count', 'unique_destination_ips', 'rdp_session_minutes', 'packet_size_variance', 'port_sweep_ratio', 'lateral_hop_count', 'signal_fusion_index', 'context_gap_score', 'stability_ratio', 'protocol_family', 'network_zone']

def standardize(series):
    return (series - series.mean()) / (series.std() + 1e-6)

print(f"Notebook: {NOTEBOOK_TITLE}")
print(f"Task type: {TASK_TYPE}")
print(f"Target column: {TARGET}")

## Step 1 - Build a Synthetic Yet Forensically Plausible Dataset

Create evidence that mirrors investigative shape while staying safe to share.

In [ ]:
def synthetic_numeric_feature(name, signal, volatility, idx, rng):
    if "entropy" in name:
        values = 4.0 + 1.2 * signal + 0.4 * volatility + rng.normal(0, 0.35, size=signal.size)
    elif any(token in name for token in ["score", "ratio", "confidence", "likelihood"]):
        values = np.clip(0.55 + 0.28 * signal + 0.18 * volatility + rng.normal(0, 0.12, size=signal.size), 0.01, 1.75)
    elif any(token in name for token in ["minutes", "hours", "latency"]):
        values = np.clip(rng.gamma(shape=2.2 + idx * 0.08, scale=4.2 + 0.9 * signal + 0.3 * volatility), 0.1, None)
    elif any(token in name for token in ["gb", "mb", "bytes"]):
        values = np.exp(2.4 + 0.3 * signal + 0.15 * volatility + rng.normal(0, 0.25, size=signal.size))
    elif any(token in name for token in ["count", "events", "edges", "hosts", "launches", "mounts"]):
        lam = np.clip(4.0 + idx * 0.35 + 2.1 * signal + 0.9 * volatility, 0.2, None)
        values = rng.poisson(lam)
    else:
        values = 35 + idx * 3 + 8.5 * signal + 3.5 * volatility + rng.normal(0, 4.0, size=signal.size)
    return np.round(values, 3)

case_complexity = RNG.normal(0, 1, N_SAMPLES)
stealth_signal = RNG.normal(0, 1, N_SAMPLES)
environment_noise = RNG.normal(0, 1, N_SAMPLES)
operational_pressure = RNG.normal(0, 1, N_SAMPLES)

if TASK_TYPE == "multiclass":
    latent_score = case_complexity + 0.45 * stealth_signal - 0.20 * environment_noise + RNG.normal(0, 0.35, N_SAMPLES)
    cluster_index = pd.qcut(latent_score, q=len(CLASS_LABELS), labels=False, duplicates="drop").astype(int)
    signal_strength = cluster_index / max(len(CLASS_LABELS) - 1, 1)
else:
    signal_strength = 0.55 * case_complexity + 0.35 * stealth_signal + 0.18 * operational_pressure + RNG.normal(0, 0.35, N_SAMPLES)

df = pd.DataFrame()
for idx, feature in enumerate(NUMERIC_FEATURES):
    df[feature] = synthetic_numeric_feature(feature, signal_strength, environment_noise, idx, RNG)

for idx, (feature, values) in enumerate(CATEGORICAL_OPTIONS.items()):
    rank = pd.Series(signal_strength + idx * 0.12 + RNG.normal(0, 0.22, N_SAMPLES)).rank(pct=True)
    bins = np.minimum((rank * len(values)).astype(int), len(values) - 1)
    preferred = np.array(values)[bins]
    random_choice = RNG.choice(values, size=N_SAMPLES)
    df[feature] = np.where(RNG.random(N_SAMPLES) < 0.24, random_choice, preferred)

if TASK_TYPE == "binary":
    risk_score = (
        1.25 * standardize(df[NUMERIC_FEATURES[0]])
        + 1.05 * standardize(df[NUMERIC_FEATURES[1]])
        + 0.95 * standardize(df[NUMERIC_FEATURES[2]])
        + 0.65 * standardize(df[NUMERIC_FEATURES[3]])
        + 0.35 * pd.Series(case_complexity)
        + 0.25 * pd.Series(operational_pressure)
    )
    for feature, values in CATEGORICAL_OPTIONS.items():
        effect_map = {value: offset for value, offset in zip(values, np.linspace(-0.35, 0.45, len(values)))}
        risk_score = risk_score + 0.12 * df[feature].map(effect_map).astype(float)
    threshold = np.quantile(risk_score, 0.78)
    df[TARGET] = np.where(risk_score > threshold, CLASS_LABELS[1], CLASS_LABELS[0])
else:
    df[TARGET] = np.array(CLASS_LABELS)[cluster_index]

df["case_id"] = [f"CASE-{i:05d}" for i in range(len(df))]
print(df.head(3).to_string(index=False))
print()
print("Dataset shape:", df.shape)
print(df[TARGET].value_counts().to_string())

## Step 2 - Evidence Profiling

Quantify the basic evidence landscape before moving to modeling.

In [ ]:
print("Numeric evidence summary:")
print(df[NUMERIC_FEATURES].describe().T[["mean", "std", "min", "max"]].round(3).to_string())
print()
print("Categorical evidence mix:")
for feature in CATEGORICAL_OPTIONS:
    print(f"\n{feature}:")
    print(df[feature].value_counts(normalize=True).round(3).to_string())

# Visualization 01: target distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
df[TARGET].value_counts(normalize=True).sort_index().plot(kind="bar", color=["#5975A4", "#CC8963", "#5F9E6E", "#C44E52"], ax=axes[0])
axes[0].set_title("Target/Class Mix")
axes[0].set_ylabel("Share")

# Visualization 02: primary feature signal
sns.histplot(data=df, x=NUMERIC_FEATURES[0], hue=TARGET, kde=True, element="step", ax=axes[1])
axes[1].set_title(f"{NUMERIC_FEATURES[0]} by target")
plt.tight_layout()

## Step 3 - Exploratory Visual Analytics

Look for structure, drift, and suspicious separation patterns visually.

In [ ]:
# Visualization 03: bivariate evidence scatter
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.scatterplot(data=df.sample(450, random_state=42), x=NUMERIC_FEATURES[0], y=NUMERIC_FEATURES[1], hue=TARGET, alpha=0.7, ax=axes[0])
axes[0].set_title("Signal separation")

# Visualization 04: correlation heatmap
corr = df[NUMERIC_FEATURES].corr().round(2)
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", ax=axes[1])
axes[1].set_title("Evidence correlation")

# Visualization 05: category-conditioned evidence
first_category = list(CATEGORICAL_OPTIONS.keys())[0]
sns.boxplot(data=df, x=first_category, y=NUMERIC_FEATURES[2], ax=axes[2], color="#88CCEE")
axes[2].tick_params(axis="x", rotation=25)
axes[2].set_title(f"{NUMERIC_FEATURES[2]} across {first_category}")
plt.tight_layout()

## Step 4 - Feature Engineering

Engineer composite forensic signals and document why each derived feature matters.

In [ ]:
df["signal_fusion_index"] = 0.45 * df["failed_auth_count"] + 0.35 * df["unique_destination_ips"] + 0.20 * df["rdp_session_minutes"]
df["context_gap_score"] = df["packet_size_variance"] - df["unique_destination_ips"]
df["stability_ratio"] = df["port_sweep_ratio"] / (1 + df["lateral_hop_count"].abs())

feature_table = pd.DataFrame(
    [
        {"engineered_feature": "signal_fusion_index", "formula": "0.45*failed_auth_count + 0.35*unique_destination_ips + 0.20*rdp_session_minutes", "forensic_rationale": "Blends failed_auth_count, unique_destination_ips, and rdp_session_minutes into a triage-ready severity index."},
        {"engineered_feature": "context_gap_score", "formula": "packet_size_variance - unique_destination_ips", "forensic_rationale": "Flags divergence between packet_size_variance and the expected context from unique_destination_ips."},
        {"engineered_feature": "stability_ratio", "formula": "port_sweep_ratio / (1 + abs(lateral_hop_count))", "forensic_rationale": "Normalizes port_sweep_ratio against volatility in lateral_hop_count."},
    ]
)
print("Feature engineering table:")
print(feature_table.to_string(index=False))

# Visualization 06: engineered feature behavior
plt.figure(figsize=(12, 4))
sns.boxplot(data=df, x=TARGET, y="signal_fusion_index", color="#88CCEE")
plt.title("Engineered feature separation")
plt.tight_layout()

## Step 5 - Baseline Benchmark

Start with a strong, explainable baseline so model lift is measurable.

In [ ]:
X = df[MODEL_FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_model_features = [col for col in MODEL_FEATURES if col not in CATEGORICAL_OPTIONS]
categorical_model_features = list(CATEGORICAL_OPTIONS.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_model_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_model_features),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2500, class_weight="balanced")),
    ]
)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_prob = baseline_model.predict_proba(X_test)
if TASK_TYPE == "binary":
    baseline_f1 = f1_score(y_test, baseline_pred, pos_label=CLASS_LABELS[1])
else:
    baseline_f1 = f1_score(y_test, baseline_pred, average="macro")
baseline_balanced_acc = balanced_accuracy_score(y_test, baseline_pred)

# Visualization 07: baseline confidence behavior
plt.figure(figsize=(12, 4))
baseline_confidence = baseline_prob.max(axis=1)
sns.histplot(baseline_confidence, bins=20, color="#4C72B0", kde=True)
plt.title("Baseline model confidence")
plt.xlabel("Predicted confidence")
plt.tight_layout()

print(f"Baseline F1: {baseline_f1:.3f}")
print(f"Baseline balanced accuracy: {baseline_balanced_acc:.3f}")

## Step 6 - Advanced Model

Train a stronger model only after the baseline establishes value.

In [ ]:
advanced_model = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=350,
                max_depth=8,
                min_samples_leaf=3,
                random_state=42,
                class_weight="balanced",
            ),
        ),
    ]
)
advanced_model.fit(X_train, y_train)
advanced_pred = advanced_model.predict(X_test)
advanced_prob = advanced_model.predict_proba(X_test)
if TASK_TYPE == "binary":
    advanced_f1 = f1_score(y_test, advanced_pred, pos_label=CLASS_LABELS[1])
else:
    advanced_f1 = f1_score(y_test, advanced_pred, average="macro")
advanced_balanced_acc = balanced_accuracy_score(y_test, advanced_pred)
print(f"Advanced F1: {advanced_f1:.3f}")
print(f"Advanced balanced accuracy: {advanced_balanced_acc:.3f}")

## Step 7 - Evaluation and Threshold Design

Review model lift, error tradeoffs, and operating metrics.

In [ ]:
report = classification_report(y_test, advanced_pred, zero_division=0, output_dict=True)
print("Classification report:")
print(classification_report(y_test, advanced_pred, zero_division=0))
positive_index = list(advanced_model.classes_).index(CLASS_LABELS[1]) if TASK_TYPE == "binary" else None

# Visualization 08: confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cm = confusion_matrix(y_test, advanced_pred, labels=advanced_model.classes_)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=advanced_model.classes_, yticklabels=advanced_model.classes_, ax=axes[0])
axes[0].set_title("Advanced model confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# Visualization 09: baseline vs advanced comparison
comparison = pd.DataFrame(
    {
        "model": ["baseline", "advanced"],
        "f1": [baseline_f1, advanced_f1],
        "balanced_accuracy": [baseline_balanced_acc, advanced_balanced_acc],
    }
)
comparison.set_index("model").plot(kind="bar", ax=axes[1], color=["#55A868", "#C44E52"])
axes[1].set_ylim(0, 1)
axes[1].set_title("Model comparison")
axes[1].set_ylabel("Score")
plt.tight_layout()

# Visualization 10: operating curve or confidence profile
if TASK_TYPE == "binary":
    advanced_auc = roc_auc_score((y_test == CLASS_LABELS[1]).astype(int), advanced_prob[:, positive_index])
    print(f"Advanced ROC-AUC: {advanced_auc:.3f}")

    precision, recall, thresholds = precision_recall_curve((y_test == CLASS_LABELS[1]).astype(int), advanced_prob[:, positive_index])
    plt.figure(figsize=(12, 4))
    plt.plot(recall, precision, color="#8172B3")
    plt.title("Precision-Recall curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.tight_layout()
else:
    advanced_auc = roc_auc_score(y_test, advanced_prob, multi_class="ovr", labels=advanced_model.classes_)
    print(f"Advanced multiclass ROC-AUC (OvR): {advanced_auc:.3f}")

    plt.figure(figsize=(12, 4))
    pd.Series(advanced_prob.max(axis=1)).plot(kind="hist", bins=20, color="#8172B3")
    plt.title("Advanced model class confidence")
    plt.xlabel("Max class probability")
    plt.tight_layout()

## Step 8 - Explainability with Feature Importance and Partial Dependence

Expose what moves predictions and how evidence values change model confidence.

In [ ]:
scoring = "balanced_accuracy" if TASK_TYPE == "binary" else "f1_macro"
permutation = permutation_importance(
    advanced_model,
    X_test,
    y_test,
    n_repeats=8,
    random_state=42,
    scoring=scoring,
)
importance_df = (
    pd.DataFrame({"feature": MODEL_FEATURES, "importance": permutation.importances_mean})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("Top signals:")
print(importance_df.head(10).to_string(index=False))

# Visualization 11: feature importance and partial dependence
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=importance_df.head(8), x="importance", y="feature", color="#4C72B0", ax=axes[0])
axes[0].set_title("Permutation feature importance")

pd_feature = next(feature for feature in MODEL_FEATURES if feature not in CATEGORICAL_OPTIONS)
grid = np.linspace(df[pd_feature].quantile(0.05), df[pd_feature].quantile(0.95), 14)
pd_values = []
for value in grid:
    X_temp = X_test.copy()
    X_temp[pd_feature] = value
    probs = advanced_model.predict_proba(X_temp)
    if TASK_TYPE == "binary":
        positive_index = list(advanced_model.classes_).index(CLASS_LABELS[1])
        pd_values.append(probs[:, positive_index].mean())
    else:
        focus_index = min(FOCUS_LABEL_INDEX, probs.shape[1] - 1)
        pd_values.append(probs[:, focus_index].mean())
axes[1].plot(grid, pd_values, color="#DD8452", linewidth=2.5)
axes[1].set_title(f"Partial dependence: {pd_feature}")
axes[1].set_xlabel(pd_feature)
axes[1].set_ylabel(f"Average P({FOCUS_LABEL})")
plt.tight_layout()

## Step 9 - Operational Triage Function

Package the notebook into a reusable analyst-facing function.

In [ ]:
numeric_fill = {feature: float(df[feature].median()) for feature in NUMERIC_FEATURES}
categorical_fill = {feature: df[feature].mode().iloc[0] for feature in CATEGORICAL_OPTIONS}

def triage_east_west_segmentation_breach_risk(evidence: dict, verbose: bool = True):
    record = pd.DataFrame([evidence]).copy()
    for feature, value in numeric_fill.items():
        if feature not in record:
            record[feature] = value
    for feature, value in categorical_fill.items():
        if feature not in record:
            record[feature] = value

    record["signal_fusion_index"] = 0.45 * record[NUMERIC_FEATURES[0]] + 0.35 * record[NUMERIC_FEATURES[1]] + 0.20 * record[NUMERIC_FEATURES[2]]
    record["context_gap_score"] = record[NUMERIC_FEATURES[3]] - record[NUMERIC_FEATURES[1]]
    record["stability_ratio"] = record[NUMERIC_FEATURES[4]] / (1 + record[NUMERIC_FEATURES[5]].abs())

    record = record[MODEL_FEATURES]
    probabilities = advanced_model.predict_proba(record)[0]
    predicted_label = advanced_model.classes_[int(np.argmax(probabilities))]
    confidence = float(np.max(probabilities))
    baseline_label = baseline_model.predict(record)[0]
    recommended_action = (
        "Escalate immediately and preserve additional evidence."
        if predicted_label == 'priority_escalation'
        else "Queue for standard review with corroborating artifact checks."
    )
    result = {
        "predicted_label": predicted_label,
        "confidence": round(confidence, 3),
        "baseline_label": baseline_label,
        "recommended_action": recommended_action,
    }
    if verbose:
        print(pd.Series(result).to_string())
    return result

example_case = {feature: float(df[feature].quantile(0.88)) for feature in NUMERIC_FEATURES}
for feature, values in CATEGORICAL_OPTIONS.items():
    example_case[feature] = values[-1]
print("triage_east_west_segmentation_breach_risk ready for analyst handoff:")
analyst_result = triage_east_west_segmentation_breach_risk(example_case, verbose=True)

# Visualization 12: analyst case vs baseline medians
comparison = pd.Series({feature: example_case[feature] - float(df[feature].median()) for feature in NUMERIC_FEATURES[:5]})
plt.figure(figsize=(12, 4))
comparison.sort_values().plot(kind="barh", color="#64B5CD")
plt.title("Example case delta from fleet median")
plt.xlabel("Difference from median")
plt.tight_layout()

## Step 10 - Summary and Investigative Handoff

Capture the key results in a compact summary table for downstream review.

In [ ]:
summary_table = pd.DataFrame(
    [
        {"dimension": "Notebook", "value": NOTEBOOK_TITLE},
        {"dimension": "Topic", "value": TOPIC_NAME},
        {"dimension": "Target", "value": TARGET},
        {"dimension": "Baseline F1", "value": round(float(baseline_f1), 3)},
        {"dimension": "Advanced F1", "value": round(float(advanced_f1), 3)},
        {"dimension": "Advanced balanced accuracy", "value": round(float(advanced_balanced_acc), 3)},
        {"dimension": "Top feature", "value": importance_df.iloc[0]['feature']},
        {"dimension": "Operational function", "value": "triage_east_west_segmentation_breach_risk"},
        {"dimension": "Primary analyst decision", "value": 'whether to isolate hosts, block destinations, or expand packet capture scope'},
    ]
)
print("Summary table:")
print(summary_table.to_string(index=False))